# 06 — Final experiment comparison and model selection

This notebook aggregates the five independently trained models. The final model is selected by maximum `validation_seen` IoU. Seen and unseen test values are reported after selection and are not used to choose the checkpoint.

The output deliberately retains negative findings: additional geometry or regularization is not assumed to improve performance.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "final_training_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "final_model").is_dir():
    raise FileNotFoundError("Run this notebook from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
print("Project root:", PROJECT_ROOT)
print("Training run:", RUN_ID)


Project root: /anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation
Training run: manual


## Controlled comparison

All rows share the same seed, data split, image resolution, frozen backbones, optimizer family, and stopping rule. The comparison chart presents validation selection separately from final test performance.

In [2]:
from final_model.training_core import build_comparison

comparison = build_comparison()
comparison

          experiment       split  samples      iou     dice  leakage  selected_epoch  validation_iou  validation_dice
baseline_object_mask   test_seen     3371 0.293547 0.394589 0.211158              14        0.281380         0.376667
baseline_object_mask test_unseen     1586 0.244231 0.335363 0.166735              14        0.281380         0.376667
           fixed_uvd   test_seen     3371 0.298380 0.398249 0.197430              12        0.285524         0.380421
           fixed_uvd test_unseen     1586 0.257012 0.350158 0.159851              12        0.285524         0.380421
     query_gated_uvd   test_seen     3371 0.295171 0.394785 0.196792              12        0.283743         0.379448
     query_gated_uvd test_unseen     1586 0.251763 0.343597 0.167633              12        0.283743         0.379448
 rotation_consistent   test_seen     3371 0.309640 0.410597 0.188260              21        0.298073         0.394854
 rotation_consistent test_unseen     1586 0.272061 0.368

,experiment,split,samples,iou,dice,leakage,selected_epoch,validation_iou,validation_dice
0,baseline_object_mask,test_seen,3371,0.293547,0.394589,0.211158,14,0.281380,0.376667
1,baseline_object_mask,test_unseen,1586,0.244231,0.335363,0.166735,14,0.281380,0.376667
2,fixed_uvd,test_seen,3371,0.298380,0.398249,0.197430,12,0.285524,0.380421
3,fixed_uvd,test_unseen,1586,0.257012,0.350158,0.159851,12,0.285524,0.380421
4,query_gated_uvd,test_seen,3371,0.295171,0.394785,0.196792,12,0.283743,0.379448
5,query_gated_uvd,test_unseen,1586,0.251763,0.343597,0.167633,12,0.283743,0.379448
6,rotation_consistent,test_seen,3371,0.309640,0.410597,0.188260,21,0.298073,0.394854
7,rotation_consistent,test_unseen,1586,0.272061,0.368372,0.153503,21,0.298073,0.394854
8,geometry_dropout,test_seen,3371,0.301276,0.402520,0.209060,15,0.287066,0.383355
9,geometry_dropout,test_unseen,1586,0.246765,0.336310,0.165282,15,0.287066,0.383355


## Deployment artifact

`training_results/best_model.pt` is a copy of the validation-selected UI checkpoint. `model_registry.json` records every candidate checkpoint, selected epoch, validation IoU, and the explicit selection rule, enabling reproducible UI comparison on unseen masks.

## Saved comparison

![Final model comparison](../final_model_comparison.png)
